# Inspect top U candidates — are they real hidden sockpuppets?

After D1+D4 with logistic regression: 32/34 features show the top-30 U users behave like confirmed sockpuppets. **But features can't distinguish "coordinated sockpuppet" from "very active legitimate user."** This notebook gathers the raw evidence so you can read it and decide.

**Outputs**:
- `top_u_inspection.csv` — one row per top-30 U user with everything you need to judge:
  - sample of their actual messages
  - threads they posted in
  - temporal overlap with P (known sockpuppets)
  - thread co-occurrence with P
  - reaction graph overlap with P

**Your job after running**: read ~20 of these by hand. After 20 you'll know whether these are sockpuppets, heavy users, or a mix.


In [ ]:
import json, re, ast
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
from pathlib import Path

# --- CONFIG ---
INPUT_TXT = "Karin_dataset/00_Datasets.txt"
FEATURES_CSV = "./user_features_pu.csv"
OUTPUT_CSV = "./top_u_inspection.csv"
TAG_FILTER = "การเมือง"
N_TOP_U = 30
N_SAMPLE_POSTS = 5      # posts per user to show in the CSV
# --------------

## Step 1: Re-derive the top-30 U candidates using LR

Using your D4 setup with logistic regression.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

feats = pd.read_csv(FEATURES_CSV)
feature_cols = [c for c in feats.columns
                if c not in ('user_id','label_3way','is_sockpuppet','ip_checked')]
X = feats[feature_cols].to_numpy(np.float32)
labels = feats['label_3way'].values
user_ids = feats['user_id'].values

P_mask = labels == 'P'
N_mask = labels == 'N'
U_mask = labels == 'U'

# Train LR on P-vs-N, score U
pn_mask = P_mask | N_mask
X_pn = X[pn_mask]
y_pn = P_mask[pn_mask].astype(int)

sc = StandardScaler().fit(X_pn)
clf = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=0)
clf.fit(sc.transform(X_pn), y_pn)

u_scores = clf.predict_proba(sc.transform(X[U_mask]))[:, 1]
u_user_ids = user_ids[U_mask]

# Top N
order = np.argsort(-u_scores)
top_user_ids = u_user_ids[order[:N_TOP_U]]
top_scores   = u_scores[order[:N_TOP_U]]

# Also keep P user IDs for coordination analysis
p_user_ids = set(user_ids[P_mask].tolist())

print(f'Top {N_TOP_U} U candidates (by LR P-vs-N score):')
for uid, s in zip(top_user_ids[:10], top_scores[:10]):
    print(f'  user_id={uid:>10}  score={s:.3f}')
print(f'  ... and {N_TOP_U - 10} more')
print(f'\nP set size: {len(p_user_ids)}')

## Step 2: Load raw posts from `00_Datasets.txt`

Apply the same tag filter as Karin's pipeline.


In [ ]:
with open(INPUT_TXT, 'r', encoding='utf-8-sig') as f:
    raw = f.read().strip()
raw = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', raw)
if raw.endswith(','): raw = raw[:-1] + '\n]'
elif not raw.endswith(']'): raw = raw + '\n]'

records = json.loads(raw, strict=False)
print(f'Loaded {len(records):,} raw records')

posts = [r for r in records if isinstance(r.get('tag'), list) and TAG_FILTER in r['tag']]
print(f'After tag filter ({TAG_FILTER!r}): {len(posts):,} posts')
del records

df = pd.DataFrame(posts)
df['dt'] = pd.to_datetime(df['datetime'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
df = df.sort_values('dt').reset_index(drop=True)

# Parse profile_like (who reacted to this post)
def to_list(v):
    if isinstance(v, list): return v
    if pd.isna(v): return []
    try:
        x = ast.literal_eval(v)
        return x if isinstance(x, list) else [x]
    except Exception: return []
df['profile_like'] = df['profile_like'].apply(to_list)

print(f'DataFrame ready: {df.shape}')

## Step 3: Per top-U user — gather everything you need to judge

For each candidate, compute:
1. **Sample of their actual messages** (longest and most-reacted)
2. **Threads they posted in** (count + list of titles)
3. **Thread co-occurrence with P users**: in how many threads did this user post alongside a confirmed sockpuppet?
4. **Temporal coordination with P**: in those shared threads, what was the minimum gap between this user's post and a P user's post? (Less than 5 minutes = strong coordination signal.)
5. **React-graph overlap with P**: do P users react to this U user's posts, or vice versa? (Coordination by mutual amplification.)


In [ ]:
# Pre-build indexes for speed
user_posts = df.groupby('user_id')
thread_users = df.groupby('thread_id')['user_id'].apply(set).to_dict()
thread_post_times = df.groupby('thread_id').apply(
    lambda g: list(zip(g['user_id'].tolist(), g['dt'].tolist()))
).to_dict()

# Who reacts to whom: build {reactor_user_id -> set(target_user_ids)}
# profile_like is on each post; entries in profile_like are users who liked that post
# So if user A's post has profile_like=[B, C], then B and C reacted to A
reacts_to = defaultdict(set)        # reactor -> set(targets)
reacted_by = defaultdict(set)       # target -> set(reactors)
for _, row in df[['user_id', 'profile_like']].iterrows():
    target = row['user_id']
    for reactor in row['profile_like']:
        if reactor != target:
            reacts_to[reactor].add(target)
            reacted_by[target].add(reactor)


def truncate(s, n=200):
    s = str(s) if s is not None else ''
    return s[:n] + ('…' if len(s) > n else '')


def inspect_user(uid):
    g = df[df['user_id'] == uid].copy()
    if len(g) == 0:
        return None

    # --- Sample messages ---
    g_sorted_len   = g.sort_values('length', ascending=False)
    g_sorted_react = g.assign(n_react=g['profile_like'].apply(len)).sort_values('n_react', ascending=False)

    longest_posts = [truncate(m) for m in g_sorted_len['messages'].head(N_SAMPLE_POSTS).tolist()]
    most_reacted  = [truncate(m) for m in g_sorted_react['messages'].head(N_SAMPLE_POSTS).tolist()]

    # --- Threads ---
    their_threads = g['thread_id'].unique().tolist()
    thread_titles_sample = g.drop_duplicates('thread_id')['thread_name'].head(8).tolist()

    # --- Thread co-occurrence with P ---
    threads_with_P = []
    for tid in their_threads:
        thread_user_set = thread_users.get(tid, set())
        ps_in_thread = thread_user_set & p_user_ids
        if ps_in_thread:
            threads_with_P.append((tid, len(ps_in_thread)))
    n_threads_with_P = len(threads_with_P)
    frac_threads_with_P = n_threads_with_P / max(len(their_threads), 1)

    # --- Temporal coordination ---
    # In each shared thread, min gap between this user's post and any P user's post
    min_gap_overall = None
    median_gap = None
    gaps = []
    for tid, _ in threads_with_P:
        events = thread_post_times.get(tid, [])
        my_times = [t for (u, t) in events if u == uid and pd.notna(t)]
        p_times  = [t for (u, t) in events if u in p_user_ids and pd.notna(t)]
        if not my_times or not p_times:
            continue
        for mt in my_times:
            for pt in p_times:
                gap = abs((mt - pt).total_seconds())
                gaps.append(gap)
    if gaps:
        min_gap_overall = min(gaps)
        median_gap = float(np.median(gaps))

    # --- React graph overlap ---
    targets_of_uid = reacts_to.get(uid, set())
    reactors_of_uid = reacted_by.get(uid, set())
    n_P_targeted_by_uid   = len(targets_of_uid & p_user_ids)
    n_P_reacting_to_uid   = len(reactors_of_uid & p_user_ids)

    return {
        'user_id': uid,
        'n_posts': len(g),
        'n_threads': len(their_threads),
        'n_threads_with_P_user': n_threads_with_P,
        'frac_threads_with_P_user': round(frac_threads_with_P, 3),
        'n_P_users_uid_reacted_to': n_P_targeted_by_uid,
        'n_P_users_who_reacted_to_uid': n_P_reacting_to_uid,
        'min_gap_sec_to_P_in_shared_thread': min_gap_overall,
        'median_gap_sec_to_P_in_shared_thread': median_gap,
        'sample_thread_titles': ' | '.join(thread_titles_sample[:5]),
        'longest_post_1': longest_posts[0] if len(longest_posts) > 0 else '',
        'longest_post_2': longest_posts[1] if len(longest_posts) > 1 else '',
        'longest_post_3': longest_posts[2] if len(longest_posts) > 2 else '',
        'most_reacted_post_1': most_reacted[0] if len(most_reacted) > 0 else '',
        'most_reacted_post_2': most_reacted[1] if len(most_reacted) > 1 else '',
    }


# Inspect each top-U user
inspection_rows = []
for uid, score in zip(top_user_ids, top_scores):
    row = inspect_user(int(uid))
    if row is None: continue
    row['lr_score'] = round(float(score), 3)
    inspection_rows.append(row)

inspection_df = pd.DataFrame(inspection_rows)
col_order = ['user_id', 'lr_score', 'n_posts', 'n_threads',
             'n_threads_with_P_user', 'frac_threads_with_P_user',
             'n_P_users_uid_reacted_to', 'n_P_users_who_reacted_to_uid',
             'min_gap_sec_to_P_in_shared_thread',
             'median_gap_sec_to_P_in_shared_thread',
             'sample_thread_titles',
             'longest_post_1','longest_post_2','longest_post_3',
             'most_reacted_post_1','most_reacted_post_2']
inspection_df = inspection_df[col_order]
print(f'Built {len(inspection_df)} rows')

## Step 4: Quick numeric summary of coordination signals

Before manual reading, look at the coordination metrics in aggregate. If the average top-U user posted in 5+ threads alongside a P user, with median time-gap under an hour, that's already strong evidence of coordination.


In [ ]:
summary = inspection_df.describe(include='all').T[['mean','50%','min','max']].rename(
    columns={'50%':'median'})
print('Coordination signals across top-30 U users:\n')
metric_cols = [
    'n_posts','n_threads','n_threads_with_P_user','frac_threads_with_P_user',
    'n_P_users_uid_reacted_to','n_P_users_who_reacted_to_uid',
    'min_gap_sec_to_P_in_shared_thread','median_gap_sec_to_P_in_shared_thread'
]
print(summary.loc[metric_cols].round(2).to_string())
print()
print('Interpretation guide:')
print('  frac_threads_with_P_user > 0.5     → mostly posts in sockpuppet-frequented threads')
print('  median_gap_sec < 3600 (1 hour)     → posts cluster with P users in time')
print('  median_gap_sec < 300 (5 min)       → strong coordination signal')
print('  P reacted to uid > 0               → mutual amplification with known sockpuppets')

## Step 5: Save the inspection CSV


In [ ]:
inspection_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f'Saved to {OUTPUT_CSV}')
print()
print('=== YOUR JOB NOW ===')
print('Open the CSV. Read user-by-user. For each, ask:')
print('  1. Do the message samples look like coordinated political messaging or')
print('     organic forum opinions?')
print('  2. Is the writing style mechanical/repetitive, or varied?')
print('  3. Do they post in threads dominated by known sockpuppets?')
print('  4. Do they reply within minutes of known sockpuppets in shared threads?')
print()
print('After reading 20 users, paste back ONE of:')
print('  A. "Most look like coordinated sockpuppets" → thesis: hidden sockpuppets found')
print('  B. "Most look like normal heavy users" → thesis: features find activity not coordination')
print('  C. "Mixed" → thesis: behavioral filtering as triage stage')

## Optional: print the top 5 users inline for quick scan

If you just want a sneak peek before opening the CSV.


In [ ]:
for i, row in inspection_df.head(5).iterrows():
    print('=' * 70)
    print(f"USER {row['user_id']} (score={row['lr_score']}, n_posts={row['n_posts']}, n_threads={row['n_threads']})")
    print(f"  Threads with known sockpuppet: {row['n_threads_with_P_user']}/{row['n_threads']} ({row['frac_threads_with_P_user']:.0%})")
    print(f"  P users this user reacted to: {row['n_P_users_uid_reacted_to']}")
    print(f"  P users who reacted to them:  {row['n_P_users_who_reacted_to_uid']}")
    gap = row['median_gap_sec_to_P_in_shared_thread']
    if pd.notna(gap):
        h = gap / 3600
        print(f"  Median time-gap to P in shared threads: {h:.1f} hours ({gap:.0f} sec)")
    print(f"  Sample threads: {row['sample_thread_titles'][:200]}")
    print(f"\n  Longest post 1: {row['longest_post_1']}")
    print(f"  Longest post 2: {row['longest_post_2']}")
    print()